In [1]:
%pip install DBcm --upgrade

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import DBcm

In [3]:
db_details = "CoachDB.sqlite3"

In [4]:
with DBcm.UseDatabase(db_details) as db:
    db.execute("pragma table_list")
    results = db.fetchall()

In [5]:
results

[('main', 'times', 'table', 4, 0, 0),
 ('main', 'events', 'table', 3, 0, 0),
 ('main', 'sqlite_sequence', 'table', 2, 0, 0),
 ('main', 'swimmers', 'table', 3, 0, 0),
 ('main', 'sqlite_schema', 'table', 5, 0, 0),
 ('temp', 'sqlite_temp_schema', 'table', 5, 0, 0)]

In [6]:
SQL = """
    select time ()
"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchall()
results    

[('13:37:39',)]

In [7]:
SQL = """
    create table if not exists swimmers (
        id integer not null primary key autoincrement,
        name varchar(32) not null,
        age integer not null
    )
"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)     

In [8]:
with DBcm.UseDatabase(db_details) as db:
    db.execute("pragma table_list")
    results = db.fetchall()
results

[('main', 'times', 'table', 4, 0, 0),
 ('main', 'events', 'table', 3, 0, 0),
 ('main', 'sqlite_sequence', 'table', 2, 0, 0),
 ('main', 'swimmers', 'table', 3, 0, 0),
 ('main', 'sqlite_schema', 'table', 5, 0, 0),
 ('temp', 'sqlite_temp_schema', 'table', 5, 0, 0)]

In [9]:
with DBcm.UseDatabase(db_details) as db:
    SQL_1 = """
        create table if not exists events ( 
            id integer not null primary key autoincrement,
            distance varchar (16) not null,
            stroke varchar(16) not null
        )
    """
    SQL_2 = """ 
        create table if not exists times (
            swimmer_id integer not null,
            event_id integer not null,
            time varchar(16) not null,
            ts timestamp default current_timestamp
        )
    """    
    db.execute(SQL_1)
    db.execute(SQL_2)

In [10]:
with DBcm.UseDatabase(db_details) as db:
    db.execute("pragma table_list")
    results = db.fetchall()
results

[('main', 'times', 'table', 4, 0, 0),
 ('main', 'events', 'table', 3, 0, 0),
 ('main', 'sqlite_sequence', 'table', 2, 0, 0),
 ('main', 'swimmers', 'table', 3, 0, 0),
 ('main', 'sqlite_schema', 'table', 5, 0, 0),
 ('temp', 'sqlite_temp_schema', 'table', 5, 0, 0)]

In [11]:
fn = 'Abi-10-100m-Back.txt'
name, age, distance, stroke = fn.removesuffix(".txt").split("-")

In [12]:
import os
FOLDER = "swimdata/"

files = os.listdir(FOLDER)
files.remove(".DS_Store")

In [13]:
import DBcm
db_details = "CoachDB.sqlite3"

In [14]:
SQL_SELECT = """
    select * from swimmers
    where name = ? and age = ?
"""

In [15]:

SQL_INSERT = """
    insert into swimmers
    (name, age)
    values
    (?, ?)
"""

In [16]:
SQL = """ select * from swimmers"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchall()
results

[(629, 'Abi', 10),
 (630, 'Ali', 12),
 (631, 'Alison', 14),
 (632, 'Aurora', 13),
 (633, 'Bill', 18),
 (634, 'Blake', 15),
 (635, 'Calvin', 9),
 (636, 'Carl', 15),
 (637, 'Chris', 17),
 (638, 'Darius', 13),
 (639, 'Dave', 17),
 (640, 'Elba', 14),
 (641, 'Emma', 13),
 (642, 'Erika', 15),
 (643, 'Hannah', 13),
 (644, 'Katie', 9),
 (645, 'Lizzie', 14),
 (646, 'Maria', 9),
 (647, 'Mike', 15),
 (648, 'Owen', 15),
 (649, 'Ruth', 13),
 (650, 'Tasmin', 15)]

In [17]:
#SQL = """delete from swimmers"""
#with DBcm.UseDatabase(db_details) as db:
 #   db.execute(SQL)
    

In [18]:
with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        name, age, _, _ = fn.removesuffix(".txt").split("-")
        db.execute(SQL_SELECT, (name, age,))
        if db.fetchall():
            continue
        db.execute(SQL_INSERT, (name, age,))

In [19]:
SQL_SELECT = """
    select * from events
    where distance = ? and stroke = ?
"""

SQL_INSERT = """
    insert into events
    (distance, stroke)
    values
    (?, ?)
"""

with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        _, _, distance, stroke = fn.removesuffix(".txt").split("-")
        db.execute(SQL_SELECT, (distance, stroke,))
        if db.fetchall():
            continue
        db.execute(SQL_INSERT, (distance, stroke,))

#   For each file the loop does this to prevent data from being inserted twice:

# db.execute(SQL_SELECT, ...) asks: “is this distance and stroke already in the table?”
# db.fetchall() returns the matching rows. It returns [] if nothing matched.
# if db.fetchall(): is true when the list has a row in it (already stored), and false when it is empty.
# continue runs only when it is true. It skips the insert and moves on to the next file.
# db.execute(SQL_INSERT, ...) runs only when the pair was not found.


In [20]:
SQL = """select * from events"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchall()
results

[(1, '100m', 'Back'),
 (2, '100m', 'Breast'),
 (3, '50m', 'Back'),
 (4, '50m', 'Breast'),
 (5, '50m', 'Free'),
 (6, '100m', 'Free'),
 (7, '200m', 'Back'),
 (8, '100m', 'Fly'),
 (9, '50m', 'Fly'),
 (10, '200m', 'IM'),
 (11, '200m', 'Breast'),
 (12, '200m', 'Free'),
 (13, '400m', 'Free')]

In [21]:
with DBcm.UseDatabase(db_details) as db:
    db.execute("pragma table_info(times)")
    results = db.fetchall()
results

[(0, 'swimmer_id', 'INTEGER', 1, None, 0),
 (1, 'event_id', 'INTEGER', 1, None, 0),
 (2, 'time', 'varchar(16)', 1, None, 0),
 (3, 'ts', 'timestamp', 0, 'current_timestamp', 0)]

In [22]:
SQL_GET_SWIMMER = """
    select id from swimmers
    where name = ? and age = ?
"""

SQL_GET_EVENT = """
    select id from events
    where distance = ? and stroke = ?
"""

SQL_INSERT_TIME = """
    insert into times
    (swimmer_id, event_id, time)
    values
    (?, ?, ?)
"""

In [27]:
with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        name, age, distance, stroke = fn.removesuffix(".txt").split("-")

        db.execute(SQL_GET_SWIMMER, (name, age,))
        s_id = db.fetchone()[0]

        db.execute(SQL_GET_EVENT, (distance, stroke,))
        e_id = db.fetchone()[0]

        with open(FOLDER + fn) as sf:
            times = sf.read().strip().split(",")

        for t in times:
            db.execute(SQL_INSERT_TIME, (s_id, e_id, t,))

In [28]:
SQL = """select count(*) from times"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchone()[0]
results

338

In [ ]:
# SQL = """delete from times"""
# with DBcm.UseDatabase(db_details) as db:
#     db.execute(SQL)
#Clear times before reloading so re-runs do not duplicate rows.